# Kick-off — Tomography of $|\alpha\rangle$ via Detuning–Phase Scans of the Analysis Train

**WP-TOM · v0.1 · 2026-05-13 · quick-and-dirty intuition check**

**Hypothesis.** A stroboscopic analysis train, scanned over its
detuning $\delta$ and AC drive phase $\varphi_\text{train}$, can
**reconstruct** a coherent displaced motional state $|\alpha\rangle$
from the resulting $P(\uparrow)(\delta, \varphi_\text{train})$ map.

**Protocol.**

1. Prepare nine input states: $|\alpha| \in \{0.1, 1, 3\}$ × $\theta_\alpha \in \{0, \pi/4, \pi/2\}$
   — "position-like", "mixed", "momentum-like" — tensored with the
   equator spin superposition $(|\uparrow\rangle + |\downarrow\rangle)/\sqrt{2}$
   (i.e. the state right after an MW $\pi/2$ pulse).
2. Apply an $N$-pulse analysis train with detuning $\delta$ and AC
   drive phase $\varphi_\text{train}$. $N \in \{1, 3, 10\}$.
3. Read out $P(\uparrow) = (1 + \langle\sigma_z\rangle)/2$ as a heatmap
   over $(\delta, \varphi_\text{train})$.

**Engine.** This notebook leans on the existing
[`scripts/stroboscopic`](../../scripts/stroboscopic) package for the
spin–motion Hamiltonian, propagators, and `StroboTrain` sequence
builder — same primitives as the
[main tutorial notebook](../../notebooks/00_tutorial_pulse_train.ipynb).


## §0 Setup

In [ ]:
import os, sys, time
import numpy as np
import matplotlib.pyplot as plt
import qutip

REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, os.path.join(REPO, 'scripts'))

from stroboscopic import HilbertSpace, operators as ops, hamiltonian as ham
from stroboscopic import propagators as prop, sequences

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.25})
print('repo:', REPO)


In [ ]:
# ── Wigner helper (same convention as the tutorial) ─────────────
def motional_rho(psi, nmax):
    down = np.asarray(psi[:nmax]); up = np.asarray(psi[nmax:])
    rho_m = np.outer(down, down.conj()) + np.outer(up, up.conj())
    return qutip.Qobj(rho_m)


def plot_wigner(ax, psi, nmax, grid=None, title=None, vmax=None):
    rho = motional_rho(psi, nmax)
    if grid is None:
        grid = np.linspace(-5.5, 5.5, 121)
    W = qutip.wigner(rho, grid, grid)
    if vmax is None:
        vmax = float(np.max(np.abs(W)))
    ax.imshow(W, extent=[grid[0], grid[-1], grid[0], grid[-1]],
              origin='lower', cmap='RdBu_r', vmin=-vmax, vmax=+vmax,
              aspect='equal')
    ax.set_xlabel(r'Re $\alpha$'); ax.set_ylabel(r'Im $\alpha$')
    ax.grid(alpha=0.15)
    if title:
        ax.set_title(title, fontsize=9)


## §1 Nine input motional states

Phase convention: $\theta_\alpha = 0$ puts $\alpha$ on $+\text{Re}$
(displacement in position), $\theta_\alpha = \pi/2$ puts it on $+\text{Im}$
(displacement in momentum), $\theta_\alpha = \pi/4$ is an equal mix.

The Wigner panels below show the motional reduced state *after* the
MW $\pi/2$ — but since the MW pulse is a pure spin rotation, the
motional marginal is unchanged: it is just a coherent state at $|\alpha|\,e^{i\theta_\alpha}$.


In [ ]:
# Hilbert space — picked generously so |α|=3 fits comfortably.
nmax = 60
hs = HilbertSpace(n_spins=1, mode_cutoffs=(nmax,))

alpha_abs_list  = [0.1, 1.0, 3.0]
alpha_phase_list_rad = [0.0, np.pi/4, np.pi/2]
alpha_phase_labels   = [r'$\theta_\alpha = 0$ (position)',
                        r'$\theta_\alpha = \pi/4$ (mix)',
                        r'$\theta_\alpha = \pi/2$ (momentum)']

psi_inputs = {}  # (i_amp, j_phase) -> psi after MW π/2
for i, a in enumerate(alpha_abs_list):
    for j, th in enumerate(alpha_phase_list_rad):
        psi0 = hs.prepare_state(
            spin={'theta_deg': 0.0, 'phi_deg': 0.0},   # |↓⟩
            modes=[{'alpha': float(a),
                    'alpha_phase_deg': float(np.degrees(th))}],
        )
        psi_inputs[(i, j)] = hs.apply_mw_pi2(psi0, mw_phase_deg=0.0)

fig, axes = plt.subplots(len(alpha_abs_list), len(alpha_phase_list_rad),
                          figsize=(9.6, 9.6))
for i, a in enumerate(alpha_abs_list):
    for j, lbl in enumerate(alpha_phase_labels):
        psi = psi_inputs[(i, j)]
        title = (lbl if i == 0 else None)
        plot_wigner(axes[i, j], psi, nmax, title=title)
        if j == 0:
            axes[i, j].set_ylabel(fr'$|\alpha|={a}$' + '\n' + r'Im $\alpha$')
plt.tight_layout(); plt.show()


## §2 Analysis-train protocol

The train is $N$ identical pulses of duration $\delta t$, repeated with
period $T_m = 2\pi/\omega_m$. Each pulse Hamiltonian (engine convention,
see [`hamiltonian.py`](../../scripts/stroboscopic/hamiltonian.py)):

$$H_\text{pulse}(\delta, \varphi_\text{train}) = \tfrac{\delta}{2}\sigma_z + \omega_m a^\dagger a + \tfrac{\Omega_r}{2}\!\left(e^{i\varphi_\text{train}} C\,\sigma_- + e^{-i\varphi_\text{train}} C^\dagger\,\sigma_+\right),$$

with $C = e^{i\eta(a + a^\dagger)}$ the Lamb–Dicke displacement
operator. After the train we measure $\langle\sigma_z\rangle$ and
report $P(\uparrow) = (1 + \langle\sigma_z\rangle)/2$.

**Drive scaling.** We rescale $\Omega_r$ per $N$ so the on-resonance
carrier rotation is always $\pi/2$ — $\Omega_\text{eff} \cdot N \cdot \delta t = \pi/2$.
This isolates the $N$-dependent comb structure from the trivial
total-rotation budget.


In [ ]:
# Physical parameters — match the tutorial baseline.
eta      = 0.397
omega_m  = 1.3
T_m      = 2 * np.pi / omega_m
dt       = 0.13 * T_m

C    = ops.coupling(eta, nmax)
Cdag = C.conj().T

def build_train(delta, ac_phase_rad, n_pulses):
    '''N-pulse train with per-N Rabi rescaling to π/2 on resonance.'''
    omega_eff = np.pi / (2 * n_pulses * dt)
    omega_r   = omega_eff / np.exp(-eta**2 / 2)
    H = ham.build_pulse_hamiltonian(
        eta, omega_r, delta, nmax, C, Cdag,
        ac_phase_rad=ac_phase_rad, omega_m=omega_m,
        intra_pulse_motion=True,
    )
    Up = prop.build_U_pulse(H, dt)
    Ug = prop.build_U_gap(nmax, omega_m, T_m - dt, delta=delta)
    return sequences.StroboTrain(U_pulse=Up, U_gap_diag=Ug, n_pulses=n_pulses)


## §3 Heatmap helper

For each $(\delta, \varphi_\text{train}, N)$ we build the train once and
reuse it across all nine input states.


In [ ]:
def scan_p_up(n_pulses, delta_grid_rel, phi_grid):
    '''Return dict (i_amp, j_phase) -> 2D array P(up) over (phi, delta) grid.'''
    P = {key: np.zeros((len(phi_grid), len(delta_grid_rel))) for key in psi_inputs}
    t0 = time.time()
    for jd, d_rel in enumerate(delta_grid_rel):
        delta = d_rel * omega_m
        for ip, phi in enumerate(phi_grid):
            tr = build_train(delta=delta, ac_phase_rad=phi, n_pulses=n_pulses)
            for key, psi0 in psi_inputs.items():
                psi_f = tr.evolve(psi0)
                obs = hs.observables(psi_f)
                P[key][ip, jd] = 0.5 * (1.0 + obs['sigma_z'])
    print(f'  N={n_pulses}: {len(delta_grid_rel) * len(phi_grid)} train builds, '
          f'{len(delta_grid_rel) * len(phi_grid) * len(psi_inputs)} evolutions '
          f'in {time.time() - t0:.1f}s')
    return P


def plot_p_up_grid(P, n_pulses, delta_grid_rel, phi_grid, vmin=0.0, vmax=1.0):
    fig, axes = plt.subplots(len(alpha_abs_list), len(alpha_phase_list_rad),
                              figsize=(11.0, 9.6), sharex=True, sharey=True)
    extent = [delta_grid_rel[0], delta_grid_rel[-1],
              phi_grid[0] / np.pi, phi_grid[-1] / np.pi]
    last = None
    for i, a in enumerate(alpha_abs_list):
        for j, lbl in enumerate(alpha_phase_labels):
            im = axes[i, j].imshow(P[(i, j)], origin='lower', extent=extent,
                                    aspect='auto', cmap='RdBu_r',
                                    vmin=vmin, vmax=vmax)
            last = im
            if i == 0:
                axes[i, j].set_title(lbl, fontsize=9)
            if j == 0:
                axes[i, j].set_ylabel(fr'$|\alpha|={a}$' + '\n' +
                                       r'$\varphi_\text{train}/\pi$')
            if i == len(alpha_abs_list) - 1:
                axes[i, j].set_xlabel(r'$\delta / \omega_m$')
    fig.suptitle(fr'$P(\uparrow)$  vs  $(\delta,\,\varphi_\text{{train}})$,  $N={n_pulses}$',
                 y=1.00, fontsize=11)
    fig.colorbar(last, ax=axes.ravel().tolist(), shrink=0.82,
                 label=r'$P(\uparrow)$')
    plt.show()


## §4 The scans

Grid: $\delta/\omega_m \in [-1.5, 1.5]$ × $\varphi_\text{train} \in [0, 2\pi)$,
21 × 21 points. Bump to 41 × 41 once you're convinced the structure
is real.


In [ ]:
delta_grid_rel = np.linspace(-1.5, 1.5, 21)
phi_grid       = np.linspace(0.0, 2 * np.pi, 21, endpoint=False)

P_N1 = scan_p_up(1,  delta_grid_rel, phi_grid)
plot_p_up_grid(P_N1, 1, delta_grid_rel, phi_grid)


In [ ]:
P_N3 = scan_p_up(3,  delta_grid_rel, phi_grid)
plot_p_up_grid(P_N3, 3, delta_grid_rel, phi_grid)


In [ ]:
P_N10 = scan_p_up(10, delta_grid_rel, phi_grid)
plot_p_up_grid(P_N10, 10, delta_grid_rel, phi_grid)


## §5 First reading — what the maps should say

Expected signatures (to be confirmed by running the cells above):

- **Empty-α floor.** For $|\alpha|=0.1$ the maps should be nearly flat
  near $P(\uparrow)=0.5$ except for a thin on-resonance dip — there is
  almost no motional state for the train to read out.
- **Ridge in $(\delta, \varphi_\text{train})$.** For $|\alpha| \geq 1$,
  ridges of $P(\uparrow)$ should appear at $\delta/\omega_m \approx \pm 1$
  (first motional sidebands), with the ridge's $\varphi_\text{train}$-offset
  rotating with $\theta_\alpha$. That offset is the tomographic angle.
- **$N$-narrowing.** At $N=1$ the structure is broad and weakly
  contrasted. At $N=10$ the ridges sharpen to width $\sim 1/N$ in
  $\delta/\omega_m$, exactly as in §1 of the
  [main tutorial](../../notebooks/00_tutorial_pulse_train.ipynb).

### Open questions for the user

These are the calls I made that you may want to redirect:

1. **Drive scaling.** I rescaled $\Omega_r$ so each $N$ delivers a π/2
   on resonance. Alternative: fix $\Omega_r$ across $N$ (per-pulse kick
   is the invariant). Which matches your intended experimental
   protocol?
2. **Phase reference for $\varphi_\text{train}$.** I took it literally
   as `ac_phase_rad` in the engine convention — no shift to compensate
   for the mid-pulse phase reference (see the
   [tutorial §4 `shift_deg`](../../notebooks/00_tutorial_pulse_train.ipynb)).
   Should the scan be shifted to match the
   [WP-E phase convention](../wp-phase-contrast-maps/)?
3. **Detuning range.** $\delta/\omega_m \in [-1.5, 1.5]$ covers the
   carrier and the first sidebands. Want to see higher orders
   (relevant for $|\alpha|=3$ where weight redistributes via the
   Jacobi–Anger fan, §2.5 of the tutorial)?
4. **Observable.** I report $P(\uparrow)$. Adding $|C| = \sqrt{\langle\sigma_x\rangle^2 + \langle\sigma_y\rangle^2}$
   alongside would distinguish "rotation along σ_z" from "loss of
   spin coherence" — useful for reading the maps. Worth adding?
